# Stage 9 — Debit spread ranking

Full-universe backtest (notebook 08, 214 instruments) showed 59.3% with a
positive avg R but high variance across instruments (std 0.21 vs mean 0.08) -
not a uniform edge. This ranks instruments the same way the other three
strategies are ranked (`strategy_scoring.py`), so the shortlist is the
instruments where this specific signal+lifecycle actually worked, not the
whole universe.

median_r_achieved is weighted highest (not avg_r_achieved) - same
median-over-mean preference as the rest of this project's stats, and it's
what caught BHARTIARTL's outlier-reliant result during validation.

In [7]:
import sys
sys.path.append("..")

import pandas as pd
from src.strategy_scoring import score_for_debit_spread, save_ranking

comparison = pd.read_excel("../exports/DEBIT_SPREAD_COMPARISON.xlsx", sheet_name="Comparison")
comparison.shape

(214, 7)

In [8]:
ranked, watchlist = score_for_debit_spread(comparison)
ranked.head(20)

,symbol,debit_spread_score,outlier_reliance_flag,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held
0,POWERINDIA,98.9,,67,64.2,0.54,1.50,5.0
1,TIINDIA,95.0,,66,54.5,0.28,0.61,5.0
2,CDSL,94.5,,68,57.4,0.41,1.46,6.0
3,INFY,93.5,,75,56.0,0.34,0.82,6.0
4,GAIL,92.7,,63,52.4,0.27,0.50,5.0
5,PFC,92.5,,61,54.1,0.36,0.95,6.0
6,SUZLON,92.1,,71,54.9,0.33,0.64,6.0
7,ADANIENSOL,91.9,,68,52.9,0.29,0.28,5.0
8,ANGELONE,91.8,,76,53.9,0.33,0.74,6.0
9,BHEL,90.9,,59,55.9,0.30,0.34,6.0


In [9]:
# How many top-20 candidates carry the outlier-reliance warning?
print(f"Instruments passing min_trades filter: {len(ranked)} of {len(comparison)}")
print(f"Flagged in top 20: {(ranked.head(20)['outlier_reliance_flag'] != '').sum()}")
ranked[ranked["outlier_reliance_flag"] != ""].head(10)

Instruments passing min_trades filter: 213 of 214
Flagged in top 20: 0


,symbol,debit_spread_score,outlier_reliance_flag,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held
39,COFORGE,76.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.38R),72,48.6,0.22,-0.16,5.0
53,DABUR,70.2,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.36R),61,49.2,0.16,-0.20,6.0
55,RADICO,69.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.35R),69,47.8,0.13,-0.22,5.0
56,MIDCPNIFTY,69.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.61R),84,48.8,0.17,-0.44,5.0
59,PNBHOUSING,68.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.37R),62,48.4,0.16,-0.21,6.0
61,CHOLAFIN,67.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.84R),67,49.3,0.20,-0.64,5.0
62,ONGC,67.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.30R),63,46.0,0.20,-0.10,6.0
63,JSWENERGY,66.9,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.70R),63,49.2,0.17,-0.53,5.0
65,GODREJPROP,66.4,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.47R),62,48.4,0.17,-0.30,6.0
66,ADANIPOWER,66.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.43R),69,46.4,0.15,-0.28,5.0


In [10]:
save_ranking(ranked, "RANKING_debit_spread.xlsx")

Saved ranking to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/RANKING_debit_spread.xlsx


'/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/RANKING_debit_spread.xlsx'

In [11]:
# Instruments below min_trades - too thin a sample to blend into the
# percentile ranking above, but not dropped either (e.g. SWIGGY: recently
# added to F&O, so it hasn't had time to generate enough signals yet).
# Sorted by avg_r_achieved so a promising-but-unproven name is visible,
# not silently missing. Re-check at the next periodic refresh - trade
# count grows on its own as more calendar time passes.
print(f"Instruments on watchlist (below min_trades): {len(watchlist)}")
watchlist

Instruments on watchlist (below min_trades): 1


,symbol,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held
0,SWIGGY,29,65.5,0.54,1.5,6.0


In [12]:
save_ranking(watchlist, "WATCHLIST_debit_spread.xlsx")

Saved ranking to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/WATCHLIST_debit_spread.xlsx


'/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/WATCHLIST_debit_spread.xlsx'